<a href="https://colab.research.google.com/github/safakatakancelik/portfolio-public/blob/main/notebooks/building_attention_from_scratch/attention_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Implementing Attention

1. Raw attention with word embeddings, no Q/K/V
2. Add Q, K, V matrices
3. Causal masking
4. Softmax
5. Full single-head attention forward pass
5. Multi-head attention (split dims across heads)
6. Training (parallel, masked) vs inference (sequential, KV cache)
7. Inspect attention matrices on ambiguous sentences
8. Rewrite as PyTorch nn.Module inside a transformer block (attention → residual → LayerNorm → FFN)

### Learning Goals
- understand raw attention vs learned Q/K/V projections
- see why √d_k scaling prevents softmax saturation
- understand causal masking for autoregressive generation
- use MHA as context for showing KV cache benefits
- compare training-time parallelism vs inference-time sequential decoding
- read attention maps to see what the model "looks at"
- bridge from numpy prototype to production-style PyTorch code


$$Attention(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

---

sources used:
https://machinelearningmastery.com/the-attention-mechanism-from-scratch/




on reasoning:

causal masking seems one of the limits of why these models are great at deductive reasoning and not abductive reasoning for example, but this is only looking at one perspective and one function. we need this functionality too.
this is based on temporal ordering

pattern matching, not reasoning: "doing sophisticated pattern matching and statistical prediction based on training data, not building an internal causal model

Passive Learning vs. Intervention

Hallucination and Spurious Correlations


---

Why Mask Instead of Truncating? Implementing not using future tokens
During training, entire sequence is processed at once in a single forward pass, not like inference, token per token.

We want to predict EVERY next token simultaneously:
this allows parallelism, and prevents immense sequential operations


In [84]:
!pip install torch
!pip install torchtext

In [85]:
import numpy as np
from scipy.special import softmax

In [86]:
np.random.seed(42)
d_model = 8
d_k = 8

### Step 1: Raw Attention

In [87]:


# building the vocabulary with random vectors
english_words = ["the", "cat", "sat", "on", "the", "mat"]

unique_words = dict.fromkeys(english_words)
vocab = {w: np.random.randn(d_model) for w in unique_words}


X = np.array([vocab[w] for w in english_words])  # (6, 8)


## Attention
attention_scores = X @ X.T / np.sqrt(d_k)  # (6, 8) @ (8, 6) -> (6, 6) attention score for each word pair

print("Tokens:", english_words)
print("\nScores shape:", attention_scores.shape)
print(np.round(attention_scores, 2))

Tokens: ['the', 'cat', 'sat', 'on', 'the', 'mat']

Scores shape: (6, 6)
[[ 2.19 -1.44 -1.61  0.08  2.19 -0.96]
 [-1.44  2.81  1.13  0.38 -1.44  1.98]
 [-1.61  1.13  2.89 -0.85 -1.61  0.37]
 [ 0.08  0.38 -0.85  2.13  0.08  0.03]
 [ 2.19 -1.44 -1.61  0.08  2.19 -0.96]
 [-0.96  1.98  0.37  0.03 -0.96  3.17]]


---


## Step 2: Add Q, K, V matrices

In [88]:
# initialize projection matrices to train
W_q = np.random.randn(d_model, d_k) * 0.01
W_k = np.random.randn(d_model, d_k) * 0.01
W_v = np.random.randn(d_model, d_k) * 0.01


# project embeddings into Q, K, V spaces
Q = X @ W_q
K = X @ W_k
V = X @ W_v

## Attention
attention_scores = Q @ K.T / np.sqrt(d_k)  # (6, 6)

print("Tokens:", english_words)
print("\nScores shape:", attention_scores.shape)
print(np.round(attention_scores, 5))

Tokens: ['the', 'cat', 'sat', 'on', 'the', 'mat']

Scores shape: (6, 6)
[[-5.80e-04  1.02e-03  2.60e-04  2.60e-04 -5.80e-04 -8.00e-05]
 [ 8.40e-04 -1.29e-03 -3.50e-04 -2.80e-04  8.40e-04 -6.10e-04]
 [-3.30e-04 -4.10e-04  5.90e-04 -3.60e-04 -3.30e-04 -2.70e-04]
 [ 2.10e-04 -2.60e-04 -3.60e-04  1.90e-04  2.10e-04 -1.30e-04]
 [-5.80e-04  1.02e-03  2.60e-04  2.60e-04 -5.80e-04 -8.00e-05]
 [ 3.40e-04 -6.80e-04  3.80e-04 -1.20e-03  3.40e-04 -1.40e-04]]


V matrix is calculated, but not in the calculation of this attenton score matrix yet.

at this stage:
$$\frac{QK^T}{\sqrt{d_k}}$$


---

## Step 3: Causal masking

In [89]:
lower_triangle_array_mask = np.tril(np.ones_like(attention_scores))
# array([[1., 0., 0., 0., 0., 0.],
#        [1., 1., 0., 0., 0., 0.],
#        [1., 1., 1., 0., 0., 0.],
#        [1., 1., 1., 1., 0., 0.],
#        [1., 1., 1., 1., 1., 0.],
#        [1., 1., 1., 1., 1., 1.]])

attention_scores[lower_triangle_array_mask == 0] = - np.inf

np.round(attention_scores, 5)

array([[-0.00058,     -inf,     -inf,     -inf,     -inf,     -inf],
       [ 0.00084, -0.00129,     -inf,     -inf,     -inf,     -inf],
       [-0.00033, -0.00041,  0.00059,     -inf,     -inf,     -inf],
       [ 0.00021, -0.00026, -0.00036,  0.00019,     -inf,     -inf],
       [-0.00058,  0.00102,  0.00026,  0.00026, -0.00058,     -inf],
       [ 0.00034, -0.00068,  0.00038, -0.0012 ,  0.00034, -0.00014]])

- causal masked applied to the attention scores before softmax
- all tokens can only attend to itself and tokens before

---

## Step 4: Softmax

In [90]:
attention_scores = softmax(attention_scores, axis=-1)

np.round(attention_scores, 5)

array([[1.     , 0.     , 0.     , 0.     , 0.     , 0.     ],
       [0.50053, 0.49947, 0.     , 0.     , 0.     , 0.     ],
       [0.33324, 0.33321, 0.33355, 0.     , 0.     , 0.     ],
       [0.25007, 0.24995, 0.24992, 0.25006, 0.     , 0.     ],
       [0.19987, 0.20019, 0.20004, 0.20004, 0.19987, 0.     ],
       [0.16675, 0.16658, 0.16676, 0.16649, 0.16675, 0.16667]])

$$\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)$$

---

## Step 5: Full single-head attention forward pass

$$Attention(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [91]:
attention_scores @ V
# (6, 6) @ (6, 8) -> (6, 8)
# back to original shape: 6 words, 8 dimensions

np.round(attention_scores, 5)

array([[1.     , 0.     , 0.     , 0.     , 0.     , 0.     ],
       [0.50053, 0.49947, 0.     , 0.     , 0.     , 0.     ],
       [0.33324, 0.33321, 0.33355, 0.     , 0.     , 0.     ],
       [0.25007, 0.24995, 0.24992, 0.25006, 0.     , 0.     ],
       [0.19987, 0.20019, 0.20004, 0.20004, 0.19987, 0.     ],
       [0.16675, 0.16658, 0.16676, 0.16649, 0.16675, 0.16667]])

---

## Step 6: Multi head

Multi head attention makes each head work on different dimensions of the vector and learn patterns in those dimensions. This is how each head specializes on different aspects.

Instead of single head processing larger vectors, learning one attention pattern, they split and get their own gradient flow. Heads naturally specialize this way unlike single head trying to learn everything.

Single head (8 dims):
Q, K, V all 8-dimensional

Multi-head (2 heads, 4 dims each):
Head 1: Q, K, V are 4-dim → learns pattern A
Head 2: Q, K, V are 4-dim → learns pattern B

The key: Each head gets its own gradient flow.
Head 1 gradients push dims [0:4] toward "syntax"
Head 2 gradients push dims [4:8] toward "semantics"

Head 1 finds a useful pattern → reduces loss
Head 2 tries different patterns → if useful, also reduces loss
If both found the same pattern → it's redundant → gradients diversify them

Previously I thought multi head attention was creating different weight projections for each head on full length. And was quite in the dark in terms of how each head specializes on different aspects. again it's all about the data and it's different dimensions.

very interesting

In [92]:
d_model = 8
num_heads = 2
d_k = d_model // num_heads

head_outputs = []

for head in range(num_heads):
  # Slice Q, K, V for this head
  start = head * d_k
  end = (head + 1) * d_k

  Q_head = Q[:, start:end]
  K_head = K[:, start:end]
  V_head = V[:, start:end]

  # computing attention for this head
  scores = Q_head @ K_head.T / np.sqrt(d_k)

  # apply mask
  mask = np.tril(np.ones_like(scores))
  scores[mask == 0] = -np.inf


  # Softmax
  weights = softmax(scores, axis=-1)

  print(f"Head {head} weights:\n{np.round(weights, 4)}\n")


  # Output
  output = weights @ V_head
  head_outputs.append(output)


# Concatenate all heads
multi_head_output = np.concatenate(head_outputs, axis=1)
# "This is your attention layer's output, ready to feed into residual connection + LayerNorm + FFN."

print(np.round(multi_head_output, 4))

Head 0 weights:
[[1.     0.     0.     0.     0.     0.    ]
 [0.5001 0.4999 0.     0.     0.     0.    ]
 [0.3333 0.3333 0.3334 0.     0.     0.    ]
 [0.2501 0.25   0.25   0.2499 0.     0.    ]
 [0.1999 0.2001 0.2    0.2001 0.1999 0.    ]
 [0.1667 0.1667 0.1667 0.1667 0.1667 0.1666]]

Head 1 weights:
[[1.     0.     0.     0.     0.     0.    ]
 [0.5007 0.4993 0.     0.     0.     0.    ]
 [0.3332 0.3332 0.3335 0.     0.     0.    ]
 [0.25   0.25   0.2499 0.2501 0.     0.    ]
 [0.1999 0.2001 0.2001 0.2    0.1999 0.    ]
 [0.1668 0.1665 0.1667 0.1664 0.1668 0.1668]]

[[-0.0173 -0.0258 -0.0094 -0.0145  0.0119 -0.0282  0.0056 -0.0352]
 [-0.0045 -0.0431 -0.005  -0.006  -0.0169 -0.0073 -0.0001 -0.0109]
 [ 0.0009 -0.0264  0.0016  0.0095 -0.0072 -0.0117  0.0054 -0.0062]
 [-0.0001 -0.0198  0.0001  0.002  -0.0089  0.0019 -0.0003 -0.0071]
 [-0.0035 -0.021  -0.0018 -0.0013 -0.0048 -0.0041  0.0009 -0.0127]
 [-0.0037 -0.0281 -0.0011 -0.0106 -0.0134 -0.0024  0.0014 -0.0066]]


In [95]:
#### review what's below to understand modify above and continue


d_model = 8
num_heads = 2
d_k = d_model // num_heads  # 4

print(f"=== SETUP ===")
print(f"d_model: {d_model}, num_heads: {num_heads}, d_k per head: {d_k}")
print(f"Original Q shape: {Q.shape}")  # (6, 8)
print(f"Original K shape: {K.shape}")  # (6, 8)
print(f"Original V shape: {V.shape}")  # (6, 8)
print()

head_outputs = []

for head in range(num_heads):
  print(f"{'='*50}")
  print(f"HEAD {head}")
  print(f"{'='*50}")

  # Slice Q, K, V for this head
  start = head * d_k  # head 0: 0, head 1: 4
  end = (head + 1) * d_k  # head 0: 4, head 1: 8
  print(f"Slicing dims [{start}:{end}]")

  Q_head = Q[:, start:end]  # (6, 4) - take 6 tokens, 4 dims
  K_head = K[:, start:end]  # (6, 4)
  V_head = V[:, start:end]  # (6, 4)
  print(f"Q_head shape: {Q_head.shape}")
  print(f"K_head shape: {K_head.shape}")
  print(f"V_head shape: {V_head.shape}")
  print()

  # Computing attention scores
  print(f"Step 1: Compute QK^T")
  K_head_T = K_head.T  # (4, 6) - transpose for multiplication
  print(f"  K_head.T shape: {K_head_T.shape}")

  QK_T = Q_head @ K_head_T  # (6, 4) @ (4, 6) = (6, 6)
  print(f"  Q_head @ K_head.T shape: {QK_T.shape}")
  print(f"  This is: 6 tokens × 6 positions (similarity scores)")
  print()

  print(f"Step 2: Scale by sqrt(d_k)")
  scores = QK_T / np.sqrt(d_k)  # (6, 6) / sqrt(4) = (6, 6) / 2.0
  print(f"  scores = QK_T / sqrt({d_k}) = QK_T / {np.sqrt(d_k):.2f}")
  print(f"  scores shape: {scores.shape}")
  print(f"  scores (raw, before mask):\n{np.round(scores, 4)}")
  print()

  # Apply causal mask
  print(f"Step 3: Apply causal mask")
  mask = np.tril(np.ones_like(scores))  # (6, 6) lower triangle = 1, upper = 0
  print(f"  mask shape: {mask.shape}")
  print(f"  mask (1=can attend, 0=future):\n{mask}")

  scores[mask == 0] = -np.inf  # Set future positions to -inf
  print(f"  scores after masking:\n{np.round(scores, 4)}")
  print()

  # Softmax
  print(f"Step 4: Apply softmax across axis=-1 (across tokens each position attends to)")
  weights = softmax(scores, axis=-1)  # (6, 6) - normalize each row
  print(f"  weights shape: {weights.shape}")
  print(f"  weights (attention probabilities, -inf → 0):\n{np.round(weights, 4)}")
  print(f"  Each row sums to 1.0 (probabilities)")
  print()

  # Multiply by values
  print(f"Step 5: Weighted sum of values")
  print(f"  weights shape: {weights.shape}")  # (6, 6)
  print(f"  V_head shape: {V_head.shape}")    # (6, 4)

  output = weights @ V_head  # (6, 6) @ (6, 4) = (6, 4)
  print(f"  weights @ V_head shape: {output.shape}")
  print(f"  This is: each of 6 tokens gets weighted sum of 6 value vectors")
  print(f"  output (6 tokens × 4 dims from this head):\n{np.round(output, 4)}")
  print()

  head_outputs.append(output)

# Concatenate all heads
print(f"{'='*50}")
print(f"CONCATENATE ALL HEADS")
print(f"{'='*50}")
print(f"Head 0 output shape: {head_outputs[0].shape}")  # (6, 4)
print(f"Head 1 output shape: {head_outputs[1].shape}")  # (6, 4)

multi_head_output = np.concatenate(head_outputs, axis=1)  # (6, 4) + (6, 4) along axis 1
print(f"Concatenating along axis=1 (horizontally)")
print(f"Final multi_head_output shape: {multi_head_output.shape}")  # (6, 8)
print(f"This is: 6 tokens × (4 from head0 + 4 from head1) = 6 tokens × 8 dims")
print()
print(f"Final output:\n{np.round(multi_head_output, 4)}")

=== SETUP ===
d_model: 8, num_heads: 2, d_k per head: 4
Original Q shape: (6, 8)
Original K shape: (6, 8)
Original V shape: (6, 8)

HEAD 0
Slicing dims [0:4]
Q_head shape: (6, 4)
K_head shape: (6, 4)
V_head shape: (6, 4)

Step 1: Compute QK^T
  K_head.T shape: (4, 6)
  Q_head @ K_head.T shape: (6, 6)
  This is: 6 tokens × 6 positions (similarity scores)

Step 2: Scale by sqrt(d_k)
  scores = QK_T / sqrt(4) = QK_T / 2.00
  scores shape: (6, 6)
  scores (raw, before mask):
[[-0.0004  0.0007  0.0001  0.0005 -0.0004 -0.0002]
 [-0.     -0.0003  0.0001 -0.0003 -0.     -0.0004]
 [-0.0001 -0.0002  0.0003  0.0003 -0.0001 -0.0011]
 [ 0.0001 -0.0004 -0.0002 -0.0004  0.0001  0.0002]
 [-0.0004  0.0007  0.0001  0.0005 -0.0004 -0.0002]
 [-0.0001 -0.0001  0.0002  0.0002 -0.0001 -0.0008]]

Step 3: Apply causal mask
  mask shape: (6, 6)
  mask (1=can attend, 0=future):
[[1. 0. 0. 0. 0. 0.]
 [1. 1. 0. 0. 0. 0.]
 [1. 1. 1. 0. 0. 0.]
 [1. 1. 1. 1. 0. 0.]
 [1. 1. 1. 1. 1. 0.]
 [1. 1. 1. 1. 1. 1.]]
  scores 